# Session 6: MongoDB Shell Methods

Diploma in AI · companion notebook

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom. In Colab, the first cell installs anything that is missing.

In [1]:
import os, sys, subprocess
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mongomock', 'pymongo<4.9'], check=False)
    # fetch helper/data files used by the examples
    base = 'https://raw.githubusercontent.com/Hamnakhan123/Diploma-in-AI/main/term-1/02-managing-large-datasets-with-mongodb/code/'
    for f in ['session-06_restock.js', 'store_db.py']:
        if not os.path.exists(f):
            subprocess.run(['wget', '-q', base + f], check=False)
print('ready')

ready


## Example: `session-06_cursor_methods.py`

In [2]:
# Session 6 · Cursor methods (mongosh -> PyMongo)
from store_db import get_db

db = get_db()

cur = db.products.find({"stock": {"$gt": 0}}, {"name": 1, "price": 1}).sort([("price", -1)]).skip(1).limit(4)
print("1) sort + skip + limit, iterated like cursor.forEach():")
for d in cur:                                  # mongosh: .forEach(d => print(d.name))
    print("  ", d["name"], d["price"])

print("\n2) next() and hasNext()-style iteration")
cur = db.customers.find({}, {"name": 1}).sort("name", 1)
first = next(cur)                              # mongosh: cur.next()
print("   first:", first["name"])
print("   remaining:", [c["name"] for c in cur])  # mongosh: while (cur.hasNext()) ...

print("\n3) map(): build a list of strings from a cursor")
labels = [f"{d['name']} - Rs {d['price']:,}" for d in db.products.find({"category": "Audio"})]
print("  ", labels)                            # mongosh: .map(d => d.name + " - Rs " + d.price)

print("\n4) toArray() + length versus count methods")
arr = list(db.orders.find({"payment": "COD"}))  # mongosh: .toArray()
print("   COD orders (toArray().length):", len(arr))
print("   countDocuments:", db.orders.count_documents({"payment": "COD"}))
print("   estimatedDocumentCount (metadata, whole collection):", db.orders.estimated_document_count())

print("\n5) batch_size(): how many documents travel per network round trip")
cur = db.orders.find().batch_size(3)           # mongosh: .batchSize(3)
print("   documents fetched:", sum(1 for _ in cur), "(4 batches of up to 3 on a real server)")

print("\n6) hint() forces a particular index")
db.products.create_index("price")
print("  ", [d["name"] for d in db.products.find({"price": {"$lt": 5000}}).hint("price_1")])

1) sort + skip + limit, iterated like cursor.forEach():
   Ultrabook 210000
   27-inch Monitor 52000
   Noise-Cancelling Headphones 18500
   Portable SSD 1TB 16000

2) next() and hasNext()-style iteration
   first: Ayesha Khan
   remaining: ['Bilal Ahmed', 'Fahad Raza', 'Hira Siddiqui', 'Sana Malik', 'Usman Tariq']

3) map(): build a list of strings from a cursor
   ['Noise-Cancelling Headphones - Rs 18,500', 'Bluetooth Speaker - Rs 6,500']

4) toArray() + length versus count methods
   COD orders (toArray().length): 4
   countDocuments: 4
   estimatedDocumentCount (metadata, whole collection): 10

5) batch_size(): how many documents travel per network round trip
   documents fetched: 10 (4 batches of up to 3 on a real server)

6) hint() forces a particular index
   ['Wireless Mouse', 'USB-C Hub', 'Webcam HD']


## Example: `session-06_collection_methods.py`

In [3]:
# Session 6 · Collection methods beyond basic CRUD
from pymongo import ReturnDocument
from store_db import get_db, show

db = get_db()

print("1) findOneAndDelete: take the oldest open (Pending/Shipped) order off a queue")
show(db.orders.find_one_and_delete({"status": {"$in": ["Pending", "Shipped"]}}, sort=[("orderDate", 1)],
                                   projection={"status": 1, "orderDate": 1}))

print("\n2) findOneAndReplace returning the new document")
show(db.customers.find_one_and_replace({"_id": 3}, {"name": "Sana Malik", "city": "Karachi", "loyalty": "Silver"},
                                       return_document=ReturnDocument.AFTER))

print("\n3) distinct with a filter")
print("   brands of products under Rs 10,000:", sorted(db.products.distinct("specs.brand", {"price": {"$lt": 10000}})))

print("\n4) createIndex / getIndexes / dropIndex")
db.products.create_index([("category", 1), ("price", -1)], name="cat_price")
print("   indexes:", list(db.products.index_information()))
db.products.drop_index("cat_price")
print("   after drop:", list(db.products.index_information()))

print("\n5) renameCollection and drop")
db.orders.aggregate([{"$match": {"status": "Cancelled"}}, {"$out": "tmp_cancelled"}])
db.tmp_cancelled.rename("cancelled_2025")
print("   collections:", sorted(db.list_collection_names()))
db.cancelled_2025.drop()
print("   after drop :", sorted(db.list_collection_names()))

1) findOneAndDelete: take the oldest open (Pending/Shipped) order off a queue
{"orderDate": "2025-03-18", "status": "Shipped", "_id": 5003}
(1 document)

2) findOneAndReplace returning the new document
{"_id": 3, "name": "Sana Malik", "city": "Karachi", "loyalty": "Silver"}
(1 document)

3) distinct with a filter
   brands of products under Rs 10,000: ['Anker', 'JBL', 'Logi', 'Redragon']

4) createIndex / getIndexes / dropIndex
   indexes: ['_id_', 'cat_price']
   after drop: ['_id_']

5) renameCollection and drop
   collections: ['cancelled_2025', 'customers', 'orders', 'products']
   after drop : ['customers', 'orders', 'products']


## Example: `session-06_restock.py`

In [4]:
# Session 6 · The same restock script in Python (compare with session-06_restock.js)
from datetime import datetime
from store_db import get_db

db = get_db()
LOW = 5
today = datetime(2025, 6, 30)                    # fixed date so the output is repeatable

print(f"Low-stock report for {db.name} on {today:%Y-%m-%d}")
for p in db.products.find({"stock": {"$lt": LOW}}, {"name": 1, "stock": 1}).sort("stock", 1):
    order = LOW * 2 - p["stock"]
    print(f"  {p['name']:<28} stock {p['stock']:>2} -> reorder {order}")
    db.reorders.insert_one({"productId": p["_id"], "qty": order, "createdAt": today})

print(list(db.reorders.aggregate([{"$group": {"_id": None, "lines": {"$sum": 1}, "units": {"$sum": "$qty"}}}])))

Low-stock report for kgs on 2025-06-30
  USB-C Hub                    stock  0 -> reorder 10
  Gaming Laptop                stock  3 -> reorder 7
[{'lines': 2, 'units': 17, '_id': None}]
